In [2]:
%load_ext autoreload
%autoreload 2


import os, json, sys, torch, math, time, timeit, random
import argparse, logging
import numpy as np
import pandas as pd
from tqdm import tqdm

from typing import IO, Any, BinaryIO, Tuple
from jaxtyping import Bool, Float, Int
from torch import Tensor

from einops import einsum

import sys, importlib
sys.path.insert(0, "/home/ruiqizhang/CS336/assignment2-systems/cs336-basics")
importlib.invalidate_caches()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from cs336_basics.model import BasicsTransformerLM
from cs336_basics.optimizer import AdamW, get_cosine_lr
from cs336_basics.nn_utils import cross_entropy

# Benchmarking

1. Benchmarking the forward and backward pass

|   num_layers |   num_heads |   d_model |   d_ff |   forward_time_mean |   forward_time_std |   backward_time_mean |   backward_time_std | model_size   |
|-------------:|------------:|----------:|-------:|--------------------:|-------------------:|---------------------:|--------------------:|:-------------|
|           12 |          12 |       768 |   3072 |           0.0448598 |         0.00191129 |             0.045304 |         0.000216567 | small        |
|           24 |          16 |      1024 |   4096 |           0.102959  |         0.0169007  |             0.160091 |         0.0249532   | medium       |
|           36 |          20 |      1280 |   5120 |           0.150725  |         0.00888408 |             0.302248 |         0.00114033  | large        |
|           48 |          25 |      1600 |   6400 |           0.493233  |         0.0149383  |             1.04276  |         0.0238563   | xl           |
|           32 |          32 |      2560 |  10240 |           0.527694  |         0.0284174  |             1.11291  |         0.0597643   | 2.7B         |

With auto-mixed precision:

|   num_layers |   num_heads |   d_model |   d_ff |   forward_time_mean |   forward_time_std |   backward_time_mean |   backward_time_std | model_size   |
|-------------:|------------:|----------:|-------:|--------------------:|-------------------:|---------------------:|--------------------:|:-------------|
|           12 |          12 |       768 |   3072 |           0.0753589 |         0.0623892  |            0.0589507 |         0.000539126 | small        |
|           24 |          16 |      1024 |   4096 |           0.0850544 |         0.00288883 |            0.10188   |         0.00217166  | medium       |
|           36 |          20 |      1280 |   5120 |           0.161667  |         0.0210455  |            0.190124  |         0.0173423   | large        |
|           48 |          25 |      1600 |   6400 |           0.209234  |         0.0245331  |            0.269029  |         0.031119    | xl           |
|           32 |          32 |      2560 |  10240 |           0.174901  |         0.0061412  |            0.27148   |         0.0431038   | 2.7B         |

2. Mixed Precision

- Max TP for the FP32 for A100s: 19.5 T Flops per second.
- Max TP for the FP16 or BP16 for A100s: 312 T Flops per second.
- Difference between those precisions:
    - FP32: 1 sign, 8-bit exponent, 23-bit mantissa (尾数位，通常用二进制). wide range + higher precision
    - FP16: 1 sign, 5‑bit exponent, 10‑bit mantissa. Narrower range and more precision than BF16.
    - BF16: 1 sign, 8‑bit exponent, 7‑bit mantissa. Same range as FP32, but low precision.
- Common techniques:
    - Loss scaling: when casting the gradient to FP16, some gradient will be too small to be represented in FP16 so they will be casted to zero, so people usually mutliply it by a constant.
    - FP16 has a smaller dynamic range so some numbers can lead to overflow. So people usually use BF16 which has the same range as FP32 and is generally stable.
- `torch.autocast(device= , dtype= )`: PyTorch’s automatic mixed‑precision context manager. 
    - It runs **eligible ops* in a lower‑precision dtype (typically FP16 or BF16) while keeping others in FP32.
    - The eligible ops are device‑specific and curated by PyTorch, mainly **heavy linear algebra (matmul, conv, GEMM, attention, etc.)**.
    - Only affect the dtype of forward pass, but not in the backward pass. All gradients are in FP32. The loss is in FP32 because the softmax operator is numerically unstable, but the logits is in FP16.
- Usually we run accumulation in FP32. See the example below.

3. Memory Profile.

- Use `torch.cuda.memory`. And we use these three lines: 

    ```
    # memory profiling.
    torch.cuda.memory._record_memory_history(max_entries=1000000)

    # save a pickle file to be loased by pytorch's online tool
    torch.cuda.memory._dump_snapshot('./results/memory_profile.pkl')

    # stop recording history.
    torch.cuda.memory._record_memory_history(enabled=None)
    ```
- Produce a pickle file and we can view this in the pytorch.org/memory_viz. An example figure looks like this:
    ![Memory Profile Example](/home/ruiqizhang/CS336/assignment2-systems/torch_memory_viz_example.png)

- Each block represents some allocation and it will show the individual allocation that was made and its size, and its stack trace.
- **Stack Trace: (栈追踪: 触发这次显存分配的函数调用链)** A stack trace is a list of function calls that were “on the call stack” at the moment an event happened (here: a CUDA memory allocation). It answers: “Which code path led to this allocation/free?”

4. Some examples of the stack trace

- The long rectangle block at the bottom (the stack trace is as below): 
    - This shows that the allocation is created at `.to(device)` in `bencmarking.py` and it calls the `.to()` fucntion in `/home/ruiqizhang/CS336/assignment2-systems/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1343:to`
    - The `/home/ruiqizhang/CS336/assignment2-systems/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:930:_apply; /home/ruiqizhang/CS336/assignment2-systems/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:903:_apply; <repeats 3 times>` means we call the two `_apply` function for three times (the `_apply` frames appears in the **call stack** for 3 times). This is because the `apply` function will be called recursively, and this module is nested in the third level.
    - This 100MB allocation is the **caching allocator's base block**, not a single tensor, so it does not have a shape. The caching allocator base block is a large chunk of GPU memory that PyTorch grabs from CUDA and then sub‑allocates into many smaller pieces for tensors. It’s a performance optimization: fewer expensive CUDA `malloc/free` calls.
        - malloc(size): asks the system (or CUDA driver on GPU) for a block of memory of size bytes.
        - free(ptr): returns that block back to the system.
        - On GPU, the equivalents are CUDA driver allocations (e.g., cudaMalloc/cudaFree). They’re relatively expensive, so PyTorch’s caching allocator reduces how often these calls happen by reusing large blocks.

```
6 Addr: b'7f537e000000_0, Size: 100.0MiB (104857600 bytes) allocation, Total memory used after allocation: 297.7MiB (312197120 bytes), Compile context: None, timestamp Sat Feb 14 2026 22:32:58 GMT-0800 (Pacific Standard Time)
CUDACachingAllocator.cpp:0:c10::cuda::CUDACachingAllocator::Native::DeviceCachingAllocator::malloc(signed char, unsigned long, CUstream_st*)
:0:c10::cuda::CUDACachingAllocator::Native::NativeCachingAllocator::malloc(void**, signed char, unsigned long, CUstream_st*)
:0:c10::cuda::CUDACachingAllocator::Native::NativeCachingAllocator::allocate(unsigned long)
:0:at::TensorBase at::detail::_empty_strided_generic<c10::ArrayRef<long> >(c10::ArrayRef<long>, c10::ArrayRef<long>, c10::Allocator*, c10::DispatchKeySet, c10::ScalarType)
??:0:at::detail::empty_strided_generic(c10::ArrayRef<long>, c10::ArrayRef<long>, c10::Allocator*, c10::DispatchKeySet, c10::ScalarType)
??:0:at::detail::empty_strided_cuda(c10::ArrayRef<long>, c10::ArrayRef<long>, c10::ScalarType, std::optional<c10::Device>)
??:0:at::detail::empty_strided_cuda(c10::ArrayRef<long>, c10::ArrayRef<long>, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>)
??:0:at::native::empty_strided_cuda(c10::ArrayRef<long>, c10::ArrayRef<long>, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>)
RegisterCUDA.cpp:0:at::(anonymous namespace)::(anonymous namespace)::wrapper_CUDA__empty_strided(c10::ArrayRef<c10::SymInt>, c10::ArrayRef<c10::SymInt>, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>)
RegisterCUDA.cpp:0:c10::impl::wrap_kernel_functor_unboxed_<c10::impl::detail::WrapFunctionIntoFunctor_<c10::CompileTimeFunctionPointer<at::Tensor (c10::ArrayRef<c10::SymInt>, c10::ArrayRef<c10::SymInt>, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>), &at::(anonymous namespace)::(anonymous namespace)::wrapper_CUDA__empty_strided>, at::Tensor, c10::guts::typelist::typelist<c10::ArrayRef<c10::SymInt>, c10::ArrayRef<c10::SymInt>, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool> > >, at::Tensor (c10::ArrayRef<c10::SymInt>, c10::ArrayRef<c10::SymInt>, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>)>::call(c10::OperatorKernel*, c10::DispatchKeySet, c10::ArrayRef<c10::SymInt>, c10::ArrayRef<c10::SymInt>, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>)
??:0:at::_ops::empty_strided::redispatch(c10::DispatchKeySet, c10::ArrayRef<c10::SymInt>, c10::ArrayRef<c10::SymInt>, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>)
RegisterBackendSelect.cpp:0:c10::impl::wrap_kernel_functor_unboxed_<c10::impl::detail::WrapFunctionIntoFunctor_<c10::CompileTimeFunctionPointer<at::Tensor (c10::ArrayRef<c10::SymInt>, c10::ArrayRef<c10::SymInt>, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>), &at::(anonymous namespace)::empty_strided>, at::Tensor, c10::guts::typelist::typelist<c10::ArrayRef<c10::SymInt>, c10::ArrayRef<c10::SymInt>, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool> > >, at::Tensor (c10::ArrayRef<c10::SymInt>, c10::ArrayRef<c10::SymInt>, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>)>::call(c10::OperatorKernel*, c10::DispatchKeySet, c10::ArrayRef<c10::SymInt>, c10::ArrayRef<c10::SymInt>, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>)
??:0:at::_ops::empty_strided::call(c10::ArrayRef<c10::SymInt>, c10::ArrayRef<c10::SymInt>, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>)
:0:at::empty_strided(c10::ArrayRef<long>, c10::ArrayRef<long>, c10::TensorOptions)
??:0:at::native::_to_copy(at::Tensor const&, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>, bool, std::optional<c10::MemoryFormat>)
RegisterCompositeExplicitAutograd.cpp:0:c10::impl::wrap_kernel_functor_unboxed_<c10::impl::detail::WrapFunctionIntoFunctor_<c10::CompileTimeFunctionPointer<at::Tensor (at::Tensor const&, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>, bool, std::optional<c10::MemoryFormat>), &at::(anonymous namespace)::(anonymous namespace)::wrapper_CompositeExplicitAutograd___to_copy>, at::Tensor, c10::guts::typelist::typelist<at::Tensor const&, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>, bool, std::optional<c10::MemoryFormat> > >, at::Tensor (at::Tensor const&, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>, bool, std::optional<c10::MemoryFormat>)>::call(c10::OperatorKernel*, c10::DispatchKeySet, at::Tensor const&, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>, bool, std::optional<c10::MemoryFormat>)
??:0:at::_ops::_to_copy::redispatch(c10::DispatchKeySet, at::Tensor const&, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>, bool, std::optional<c10::MemoryFormat>)
RegisterBackendSelect.cpp:0:c10::impl::wrap_kernel_functor_unboxed_<c10::impl::detail::WrapFunctionIntoFunctor_<c10::CompileTimeFunctionPointer<at::Tensor (at::Tensor const&, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>, bool, std::optional<c10::MemoryFormat>), &at::(anonymous namespace)::_to_copy>, at::Tensor, c10::guts::typelist::typelist<at::Tensor const&, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>, bool, std::optional<c10::MemoryFormat> > >, at::Tensor (at::Tensor const&, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>, bool, std::optional<c10::MemoryFormat>)>::call(c10::OperatorKernel*, c10::DispatchKeySet, at::Tensor const&, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>, bool, std::optional<c10::MemoryFormat>)
??:0:at::_ops::_to_copy::redispatch(c10::DispatchKeySet, at::Tensor const&, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>, bool, std::optional<c10::MemoryFormat>)
VariableType_0.cpp:0:torch::autograd::VariableType::(anonymous namespace)::_to_copy(c10::DispatchKeySet, at::Tensor const&, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>, bool, std::optional<c10::MemoryFormat>)
VariableType_0.cpp:0:c10::impl::wrap_kernel_functor_unboxed_<c10::impl::detail::WrapFunctionIntoFunctor_<c10::CompileTimeFunctionPointer<at::Tensor (c10::DispatchKeySet, at::Tensor const&, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>, bool, std::optional<c10::MemoryFormat>), &torch::autograd::VariableType::(anonymous namespace)::_to_copy>, at::Tensor, c10::guts::typelist::typelist<c10::DispatchKeySet, at::Tensor const&, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>, bool, std::optional<c10::MemoryFormat> > >, at::Tensor (c10::DispatchKeySet, at::Tensor const&, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>, bool, std::optional<c10::MemoryFormat>)>::call(c10::OperatorKernel*, c10::DispatchKeySet, at::Tensor const&, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>, bool, std::optional<c10::MemoryFormat>)
??:0:at::_ops::_to_copy::call(at::Tensor const&, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>, bool, std::optional<c10::MemoryFormat>)
??:0:at::native::to(at::Tensor const&, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>, bool, bool, std::optional<c10::MemoryFormat>)
RegisterCompositeImplicitAutograd.cpp:0:c10::impl::wrap_kernel_functor_unboxed_<c10::impl::detail::WrapFunctionIntoFunctor_<c10::CompileTimeFunctionPointer<at::Tensor (at::Tensor const&, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>, bool, bool, std::optional<c10::MemoryFormat>), &at::(anonymous namespace)::(anonymous namespace)::wrapper_CompositeImplicitAutograd_dtype_layout_to>, at::Tensor, c10::guts::typelist::typelist<at::Tensor const&, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>, bool, bool, std::optional<c10::MemoryFormat> > >, at::Tensor (at::Tensor const&, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>, bool, bool, std::optional<c10::MemoryFormat>)>::call(c10::OperatorKernel*, c10::DispatchKeySet, at::Tensor const&, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>, bool, bool, std::optional<c10::MemoryFormat>)
??:0:at::_ops::to_dtype_layout::call(at::Tensor const&, std::optional<c10::ScalarType>, std::optional<c10::Layout>, std::optional<c10::Device>, std::optional<bool>, bool, bool, std::optional<c10::MemoryFormat>)
python_variable_methods.cpp:0:torch::autograd::dispatch_to(at::Tensor const&, c10::Device, bool, bool, std::optional<c10::MemoryFormat>)
python_variable_methods.cpp:0:torch::autograd::THPVariable_to(_object*, _object*, _object*)
??:0:method_vectorcall_VARARGS_KEYWORDS.llvm.8178272430361517305
/home/ruiqizhang/CS336/assignment2-systems/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1329:convert
??:0:run_mod.llvm.29710523149983642
??:0:pyrun_file
??:0:_PyRun_SimpleFileObject
??:0:_PyRun_AnyFileObject
??:0:pymain_run_file_obj
??:0:pymain_run_file
??:0:Py_RunMain
??:0:pymain_main
??:0:Py_BytesMain
??:0:__libc_init_first
/home/ruiqizhang/CS336/assignment2-systems/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:930:_apply
/home/ruiqizhang/CS336/assignment2-systems/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:903:_apply
<repeats 3 times>
/home/ruiqizhang/CS336/assignment2-systems/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1343:to
/home/ruiqizhang/CS336/assignment2-systems/./scripts/benchmarking.py:129:benchmarking
/home/ruiqizhang/CS336/assignment2-systems/./scripts/benchmarking.py:242:<module>
```

In [29]:
print([n for n in dir(torch._C) if "autocast" in n.lower()])

x = torch.randn(4, 4, device="cuda", dtype=torch.float32)
with torch.autocast(device_type="cuda", dtype=torch.float16):
    y1 = torch.matmul(x, x)
    y2 = x @ x
    y3 = x.pow(2)
    y4 = x.sum()
    y5 = x * x
    y6 = x.mean()
    y7 = x.norm()
    y8 = torch.softmax(x, dim=-1)
    y9 = torch.exp(x)
    y10 = torch.log(x)
    y11 = torch.mm(x, x)

print(y1.dtype, y2.dtype, y3.dtype, y4.dtype, y5.dtype)  # fp16 means eligible; fp32 means not cast
print(y6.dtype, y7.dtype, y8.dtype, y9.dtype, y10.dtype)
print(y11.dtype)

['_DisableAutocast', '_is_any_autocast_enabled', '_is_autocast_available', '_jit_pass_autocast', '_jit_set_autocast_mode', 'autocast_decrement_nesting', 'autocast_increment_nesting', 'clear_autocast_cache', 'get_autocast_cpu_dtype', 'get_autocast_dtype', 'get_autocast_gpu_dtype', 'get_autocast_ipu_dtype', 'get_autocast_xla_dtype', 'is_autocast_cache_enabled', 'is_autocast_cpu_enabled', 'is_autocast_enabled', 'is_autocast_ipu_enabled', 'is_autocast_xla_enabled', 'set_autocast_cache_enabled', 'set_autocast_cpu_dtype', 'set_autocast_cpu_enabled', 'set_autocast_dtype', 'set_autocast_enabled', 'set_autocast_gpu_dtype', 'set_autocast_ipu_dtype', 'set_autocast_ipu_enabled', 'set_autocast_xla_dtype', 'set_autocast_xla_enabled']
torch.float16 torch.float16 torch.float32 torch.float32 torch.float32
torch.float32 torch.float32 torch.float32 torch.float32 torch.float32
torch.float16


In [31]:
# accumulation

s = torch.tensor(0, dtype = torch.float32)
for i in range(1000):
    s += torch.tensor(0.01, dtype = torch.float32)
print(s)

s = torch.tensor(0, dtype = torch.float16)
for i in range(1000):
    s += torch.tensor(0.01, dtype = torch.float16)
print(s)

s = torch.tensor(0, dtype = torch.float32)
for i in range(1000):
    s += torch.tensor(0.01, dtype = torch.float16)
print(s)

s = torch.tensor(0, dtype = torch.float16)
for i in range(1000):
    s += torch.tensor(0.01, dtype = torch.float32)
print(s)

tensor(10.0001)
tensor(9.9531, dtype=torch.float16)
tensor(10.0021)
tensor(9.9531, dtype=torch.float16)


In [ ]:
from scripts.model_toy import ToyModel

model = ToyModel(in_features=10, out_features=10)
device = 'cuda:7'
model.to(device)
dtype = torch.float16

with torch.autocast(device_type='cuda', dtype=dtype):
    x = torch.randn(64, 10, device=device)
    logits = model(x, verbose=True)
    print(f'logits.dtype: {logits.dtype}')

    loss = torch.nn.functional.cross_entropy(logits, torch.randint(0, 10, (64,), device=device))
    print(f'loss.dtype: {loss.dtype}')
    loss.backward()
    print(f'loss.dtype after backward: {loss.dtype}')

    for name, param in model.named_parameters():
        if param.grad is not None:
            print(f"{name} grad dtype: {param.grad.dtype}")

input dtype: torch.float32
fc1 weight dtype: torch.float32
ln1 weight dtype: torch.float32
fc2 weight dtype: torch.float32
fc1 output dtype: torch.float16
relu output dtype: torch.float16
ln1 output dtype: torch.float32
fc2 output dtype: torch.float16
logits.dtype: torch.float16
loss.dtype: torch.float32
loss.dtype after backward: torch.float32
fc1.weight grad dtype: torch.float32
ln1.weight grad dtype: torch.float32
ln1.bias grad dtype: torch.float32
fc2.weight grad dtype: torch.float32


In [ ]:
from scripts.model_toy import ToyModel

model = ToyModel(in_features=10, out_features=10)
device = 'cuda:7'
model.to(device)
dtype = torch.bfloat16

with torch.autocast(device_type='cuda', dtype=dtype):
    x = torch.randn(64, 10, device=device)
    logits = model(x, verbose=True)
    print(f'logits.dtype: {logits.dtype}')

    loss = torch.nn.functional.cross_entropy(logits, torch.randint(0, 10, (64,), device=device))
    print(f'loss.dtype: {loss.dtype}')
    loss.backward()
    print(f'loss.dtype after backward: {loss.dtype}')

    for name, param in model.named_parameters():
        if param.grad is not None:
            print(f"{name} grad dtype: {param.grad.dtype}")


input dtype: torch.float32
fc1 weight dtype: torch.float32
ln1 weight dtype: torch.float32
fc2 weight dtype: torch.float32
fc1 output dtype: torch.bfloat16
relu output dtype: torch.bfloat16
ln1 output dtype: torch.float32
fc2 output dtype: torch.bfloat16
logits.dtype: torch.bfloat16
loss.dtype: torch.float32
loss.dtype after backward: torch.float32
fc1.weight grad dtype: torch.float32
ln1.weight grad dtype: torch.float32
ln1.bias grad dtype: torch.float32
fc2.weight grad dtype: torch.float32


5. Memory profiling result.

We use 2.7B model and we profile the memory used by outputting the memory snapshot in a pkl file, and also output the peak memory by calling `torch.cuda.max_memory_allocated()`. We try `T = 128, 256, 512` and we profile it with and without AMP. 

- The peak memory does not scale with T very much especially when T is 128 and 256, since the memory is taken primarly by the P+G+O, not the activation.

- 

| mixed-precision | T | forward-only | peak memory |
|-----------------|---|--------------|-------------|
|        True         | 128  |     True         |     12.9GB        | 
|        True        | 256 |      True        |    13.0GB        | 
|        True         | 512 |     True         |    13.5GB        |   
|        True        | 128 |      False        |    51.4GB        |    
|        True         | 256 |     False         |    51.4GB       |  
|        True         | 512 |     False         |   65.5GB        |    
|        False         | 128 |    True          |   19.1GB          | 
|        False         | 256 |     True         |   19.2GB          |  
|        False         | 512 |    True          |    19.4GB         |  
|        False         | 128 |    False          |   51.4GB          | 
|        False         | 256 |    False          |   52.1GB          |  
|        False         | 512 |    False          |   62.7GB          | 



In [10]:
# theoretical activation memory
# Fixed memory = 16 N  Bytes.
# For 2.7B model, this is 43.2 GB.

V = 10000
D = 2560
L = 32
H = 32
D_ff = 10240
B = 4

for T in [128, 256, 512]:
    A =  4 * B * T * ((12*L+1)*D + (2*L+1) + V + H*L)
    print(43.2 + A / 1024**3)

45.10103340148926
47.00206680297852
50.804133605957034
